In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams.update({
    'font.size': 22,
    'axes.titlesize': 22,
    'axes.labelsize': 22,
    'xtick.labelsize': 22,
    'ytick.labelsize': 22,
    'legend.fontsize': 22,
    'figure.titlesize': 22
})

def load_and_preprocess_single_site(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found at {filepath}")
    
    data = pd.read_csv(filepath, dtype=str, low_memory=False)
    data = data.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
    data.replace({'': np.nan, 'NaN': np.nan, 'NaT': np.nan}, inplace=True)
    
    if 'Date' not in data.columns or 'Time' not in data.columns:
        raise ValueError("CSV file must contain 'Date' and 'Time' columns")
    
    def fix_time(t):
        return '00:00:00' if t == '24:00:00' else t
    
    data['Time'] = data['Time'].apply(fix_time)
    
    data['DateTime'] = pd.to_datetime(
        data['Date'] + ' ' + data['Time'],
        dayfirst=True,
        format='mixed',
        errors='coerce'
    )
    
    data = data.dropna(subset=['DateTime'])
    data.drop(['Date', 'Time'], axis=1, inplace=True)
    
    data_columns = [c for c in data.columns if c != 'DateTime']
    if len(data_columns) == 0:
        raise ValueError("No data column found")
    
    data_column = data_columns[0]
    site_name = os.path.splitext(os.path.basename(filepath))[0]
    
    temp_df = data[['DateTime', data_column]].copy()
    
    value_extract = temp_df[data_column].str.extract(r'^([\d\.-]+)')
    temp_df['Value'] = pd.to_numeric(value_extract[0], errors='coerce')
    
    temp_df = temp_df.dropna(subset=['Value'])
    temp_df = temp_df[temp_df['Value'] > 0]
    
    temp_df['Location'] = site_name
    
    processed_data = temp_df[['DateTime', 'Location', 'Value']]
    
    print(f"Loaded {len(processed_data)} records for {site_name}")
    print(f"Date range: {processed_data['DateTime'].min()} → {processed_data['DateTime'].max()}")
    
    return processed_data

def plot_concentration_time(data, location_name):    
    df = data[data['Location'] == location_name].sort_values('DateTime')
    
    plt.figure(figsize=(16, 8))
    
    plt.plot(
        df['DateTime'],
        df['Value'],
        linewidth=0.8,
        color='tab:blue'
    )
    
    plt.xlabel('Time')
    plt.ylabel('NO₂ Concentration (µg/m³)')
    plt.title(f'NO₂ Concentration vs Time – {location_name}', fontweight='bold')
    
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    plt.savefig(
        r"C:\Users\qp24695\Documents\NO2_TimeSeries.png",
        dpi=300,
        bbox_inches='tight'
    )
    
    plt.show()

def analyze_location(data, location_name):

    location_data = data[data['Location'] == location_name]
    
    if len(location_data) < 10:
        print(f"Not enough data for {location_name}")
        return
    
    print(f"\nPlotting time series for {location_name}")
    print(f"Observations: {len(location_data)}")
    
    plot_concentration_time(data, location_name)

def main():
    filepath = r"C:\Users\qp24695\Downloads\.csv"
    
    data = load_and_preprocess_single_site(filepath)
    
    locations = data['Location'].unique()
    print(f"\nFound locations: {locations}")
    
    for location in locations:
        analyze_location(data, location)


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, minimize
from scipy.integrate import quad
import os

plt.rcParams.update({
    'font.size': 48,
    'axes.titlesize': 52,
    'axes.labelsize': 50,
    'xtick.labelsize': 48,
    'ytick.labelsize': 48,
    'legend.fontsize': 52,
    'figure.titlesize': 62
})

def q_chi_pdf(x, a, q, lambda_):
    exponent = 1 / (1 - q + 1e-10)  
    term = 1 - (1 - q) * lambda_ * x
    term = np.clip(term, 1e-10, None)  
    return (x**(a-1)) * (term ** exponent)

def q_chi_cdf(x, a, q, lambda_, x_min, x_max):
    try:
        cdf_values = np.zeros_like(x)
        norm_const = calculate_norm_const(a, q, lambda_, x_min, x_max)
        if not np.isfinite(norm_const) or norm_const <= 0:
            return np.full_like(x, np.nan)
        for i, x_val in enumerate(x):
            if x_val <= x_min:
                cdf_values[i] = 0
            elif x_val >= x_max:
                cdf_values[i] = 1
            else:
                integral, _ = quad(lambda t: q_chi_pdf(t, a, q, lambda_),
                                 x_min, x_val,
                                 limit=200, epsabs=1e-8, epsrel=1e-8)
                cdf_values[i] = integral / norm_const
        return cdf_values
    except:
        return np.full_like(x, np.nan)

def calculate_norm_const(a, q, lambda_, x_min, x_max):
    try:
        integral, error = quad(lambda x: q_chi_pdf(x, a, q, lambda_),
                             x_min, x_max,
                             limit=200, epsabs=1e-8, epsrel=1e-8)
        if integral <= 0 or not np.isfinite(integral):
            return np.inf
        return integral
    except:
        return np.inf

def neg_log_likelihood(params, data):
    a, q, lambda_ = params
    penalty = 0
    if q <= 0 or q >= 2:
        penalty += 1e6 * (abs(q - 1) + 1)  
    pdf_values = q_chi_pdf(data, a, q, lambda_)
    x_range = np.linspace(min(data), max(data), 1000)
    pdf_norm = q_chi_pdf(x_range, a, q, lambda_)
    norm = np.trapz(pdf_norm, x_range)
    epsilon = 1e-10
    normalized_pdf = pdf_values / (norm + epsilon)
    log_pdf = np.log(normalized_pdf + epsilon)
    return -np.sum(log_pdf) + penalty

def fit_distribution_mle(values):
    data_mean = np.mean(values)
    initial_lambda = 1.0 / data_mean if data_mean > 0 else 1.0
    initial_params = [2.0, 1.0, initial_lambda]   
    bounds = [(0.001, 15), (0, 2), (0.001, 150)]  

    result = minimize(neg_log_likelihood, initial_params, args=(values,),
                      bounds=bounds, method='L-BFGS-B',
                      options={'maxiter': 1000, 'ftol': 1e-8, 'gtol': 1e-6})

    if not result.success or result.fun > 1e9:
        initial_guesses = [
            [1.5, 0.8, 0.5],
            [2.5, 1.5, 2.0],
            [3.0, 0.5, 1.0],
            [5.0, 1.8, 0.1],
            [0.5, 1.2, 10.0],
            [10.0, 0.2, 50.0]
        ]
        best_result = result
        for guess in initial_guesses:
            try:
                result_temp = minimize(neg_log_likelihood, guess, args=(values,),
                                       bounds=bounds, method='L-BFGS-B',
                                       options={'maxiter': 1000, 'ftol': 1e-8})
                if result_temp.fun < best_result.fun and result_temp.success:
                    best_result = result_temp
            except:
                continue
        result = best_result

    if result.success and np.isfinite(result.fun) and result.fun < 1e9:
        a, q, lambda_ = result.x
        if (bounds[0][0] <= a <= bounds[0][1] and
            bounds[1][0] <= q <= bounds[1][1] and
            bounds[2][0] <= lambda_ <= bounds[2][1]):
            print(f"MLE success for n={len(values)}: a={a:.3f}, q={q:.3f}, lambda={lambda_:.3f}, NLL={result.fun:.3f}")
            return result.x

    print(f"MLE warning: Optimization may not have converged properly")
    return result.x

def debug_mle_fit(values, params):
    a, q, lambda_ = params
    print(f"Debug - Parameters: a={a:.4f}, q={q:.4f}, lambda={lambda_:.4f}")
    x_min = max(0, np.min(values) * 0.5)
    x_max = np.max(values) * 2
    norm_const = calculate_norm_const(a, q, lambda_, x_min, x_max)
    print(f"Debug - Normalization constant: {norm_const:.6f}")
    pdf_values = q_chi_pdf(values, a, q, lambda_) / norm_const
    valid_ratio = np.sum(pdf_values > 1e-12) / len(pdf_values)
    print(f"Debug - Valid PDF values ratio: {valid_ratio:.4f}")
    nll = neg_log_likelihood(params, values)
    print(f"Debug - Negative log-likelihood: {nll:.4f}")
    return nll

def fit_distribution_curve_fit(values, num_bins=200):
    hist, bin_edges = np.histogram(values, bins=num_bins, density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    mask = hist > 0
    hist = hist[mask]
    bin_centers = bin_centers[mask]
    x_min = max(0, np.min(values) * 0.5)
    x_max = np.max(values) * 2

    def normalized_pdf_wrapper(x, a, q, lambda_):
        norm_const = calculate_norm_const(a, q, lambda_, x_min, x_max)
        if not np.isfinite(norm_const) or norm_const <= 0:
            return np.full_like(x, 1e10)  
        return q_chi_pdf(x, a, q, lambda_) / norm_const

    bounds = ([0.001, 0.001, 0.001], [15, 2.5, 150])
    data_mean = np.mean(values)
    initial_lambda = 1.0 / data_mean if data_mean > 0 else 1.0

    try:
        popt_chi, pcov = curve_fit(
            normalized_pdf_wrapper,
            bin_centers,
            hist,
            p0=[2.0, 1.5, initial_lambda],
            bounds=bounds,
            maxfev=10000
        )
        print(f"Curve_fit success: a={popt_chi[0]:.3f}, q={popt_chi[1]:.3f}, lambda={popt_chi[2]:.3f}")
    except RuntimeError as e:
        print(f"Failed to fit distribution with curve_fit: {e}")
        return None, None, None, None
    except Exception as e:
        print(f"Unexpected error in curve_fit: {e}")
        return None, None, None, None

    return popt_chi, bin_centers, bin_edges, hist

def perform_anderson_darling_test(values, popt_chi):
    if popt_chi is None:
        return np.nan, np.nan
    try:
        sorted_data = np.sort(values)
        n = len(sorted_data)
        if n < 5:
            print("Warning: Not enough data points for Anderson-Darling test")
            return np.nan, np.nan
        x_min = max(0, np.min(values) * 0.5)
        x_max = np.max(values) * 2
        cdf_values = q_chi_cdf(sorted_data, *popt_chi, x_min, x_max)
        if np.any(np.isnan(cdf_values)) or np.any(cdf_values < 0) or np.any(cdf_values > 1):
            print("Warning: Invalid CDF values for Anderson-Darling test")
            return np.nan, np.nan
        i = np.arange(1, n + 1)
        term1 = (2 * i - 1) * np.log(cdf_values)
        term2 = (2 * (n - i) + 1) * np.log(1 - cdf_values)
        A2 = -n - np.sum(term1 + term2) / n
        
        if A2 <= 0.5:
            p_value_approx = 0.9
        elif A2 <= 0.75:
            p_value_approx = 0.7
        elif A2 <= 1.0:
            p_value_approx = 0.5
        elif A2 <= 1.5:
            p_value_approx = 0.25
        elif A2 <= 2.0:
            p_value_approx = 0.1
        else:
            p_value_approx = 0.01
        return A2, p_value_approx
    except Exception as e:
        print(f"Error in Anderson-Darling test: {e}")
        return np.nan, np.nan

def perform_goodness_of_fit(values, popt_chi):
    if popt_chi is None:
        return {}
    results = {}
    ad_stat, ad_p = perform_anderson_darling_test(values, popt_chi)
    results['anderson_darling'] = {'statistic': ad_stat, 'p_value': ad_p}
    return results

def load_and_preprocess_multipollutant(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found at {filepath}")

    ext = os.path.splitext(filepath)[1].lower()

    try:
        if ext in ['.xlsx', '.xls']:
            data = pd.read_excel(filepath, dtype=str)
        else:
            try:
                data = pd.read_csv(filepath, dtype=str, low_memory=False, encoding='utf-8')
            except UnicodeDecodeError:
                try:
                    data = pd.read_csv(filepath, dtype=str, low_memory=False, encoding='latin1')
                except UnicodeDecodeError:
                    data = pd.read_csv(filepath, dtype=str, low_memory=False, encoding='cp1252')

        data = data.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
        data.replace({'': np.nan, 'NaN': np.nan, 'NaT': np.nan}, inplace=True)

    except Exception as e:
        raise ValueError(f"Error reading file: {str(e)}")

    if 'Date' not in data.columns or 'Time' not in data.columns:
        raise ValueError("CSV file must contain 'Date' and 'Time' columns")

    def fix_time(time_str):
        if time_str == '24:00:00':
            return '00:00:00'
        return time_str
    data['Time'] = data['Time'].apply(fix_time)

    try:
        datetime_str = data['Date'] + ' ' + data['Time']
        data['DateTime'] = pd.to_datetime(
            datetime_str,
            dayfirst=True,
            format='mixed',
            errors='coerce'
        )
        data = data.dropna(subset=['DateTime'])
        data.drop(['Date', 'Time'], axis=1, inplace=True)
    except Exception as e:
        raise ValueError(f"Error processing datetime: {str(e)}")

    pollutant_pairs = {
        'Nitric oxide': 'Status',
        'Nitrogen dioxide': 'Status',
        'PM10 particulate matter': 'Status',
        'PM2.5 particulate matter': 'Status',
        'Ozone':'Status',
    }
    value_columns = [col for col in data.columns if col in pollutant_pairs.keys()]
    if not value_columns:
        raise ValueError("No recognized pollutant columns found. Expected: Nitric oxide, Nitrogen dioxide, PM10 particulate matter, PM2.5 particulate matter, Ozone")

    col_list = list(data.columns)
    records = []
    site_name = os.path.splitext(os.path.basename(filepath))[0]  

    for val_col in value_columns:
        idx = col_list.index(val_col)
        if idx + 1 < len(col_list):
            status_col = col_list[idx + 1]
        else:
            print(f"Warning: No status column found after {val_col}. Skipping.")
            continue

        temp_df = data[['DateTime', val_col, status_col]].copy()
        temp_df.columns = ['DateTime', 'Value', 'Status']  

        temp_df['Value'] = pd.to_numeric(temp_df['Value'], errors='coerce')
        temp_df = temp_df.dropna(subset=['Value'])
        temp_df = temp_df[temp_df['Value'] > 0]

        if len(temp_df) == 0:
            print(f"Warning: No valid positive data for {val_col}")
            continue

        temp_df['Pollutant'] = val_col
        temp_df['Location'] = site_name

        records.append(temp_df[['DateTime', 'Location', 'Pollutant', 'Value', 'Status']])

    if not records:
        raise ValueError("No valid data found for any pollutant.")

    combined = pd.concat(records, ignore_index=True)

    print(f"Successfully loaded data for site: {site_name}")
    print(f"Total records: {len(combined)}")
    print(f"Pollutants found: {combined['Pollutant'].unique()}")
    print(f"Date range: {combined['DateTime'].min()} to {combined['DateTime'].max()}")

    return combined

def plot_on_axis(ax, values, bin_edges, popt_curve_fit, popt_mle,
                 pollutant_name, gof_results_cf, gof_results_mle,
                 xscale='linear', yscale='linear', title_suffix=''):
    
    ax.tick_params(axis='both', labelsize=54)

    if xscale == 'log':
        x_fit = np.logspace(np.log10(max(bin_edges[0], 1e-6)),
                            np.log10(bin_edges[-1]), 500)
    else:
        x_fit = np.linspace(bin_edges[0], bin_edges[-1], 500)

    n, bins, patches = ax.hist(values, bins=bin_edges, density=True, alpha=0.6,
                                label='Histogram', color='skyblue', edgecolor='black')

    if popt_curve_fit is not None:
        x_min = max(0, np.min(values) * 0.5)
        x_max = np.max(values) * 2
        norm_const_cf = calculate_norm_const(*popt_curve_fit, x_min, x_max)
        if np.isfinite(norm_const_cf) and norm_const_cf > 0:
            y_fit_cf = q_chi_pdf(x_fit, *popt_curve_fit) / norm_const_cf
            label_cf = f"SSR: α={popt_curve_fit[0]:.2f}, q={popt_curve_fit[1]:.2f}, λ={popt_curve_fit[2]:.2f}"
            ax.plot(x_fit, y_fit_cf, 'r--', linewidth=3, label=label_cf)

    if popt_mle is not None:
        x_min = max(0, np.min(values) * 0.5)
        x_max = np.max(values) * 2
        norm_const = calculate_norm_const(*popt_mle, x_min, x_max)
        if np.isfinite(norm_const) and norm_const > 0:
            y_fit_mle = q_chi_pdf(x_fit, *popt_mle) / norm_const
            label_mle = f"MLE: α={popt_mle[0]:.2f}, q={popt_mle[1]:.2f}, λ={popt_mle[2]:.2f}"
            ax.plot(x_fit, y_fit_mle, 'g-', linewidth=3, label=label_mle)

    ax.set_xscale(xscale)
    ax.set_yscale(yscale)

    full_title = f"{pollutant_name}" + (f" ({title_suffix})" if title_suffix else "")
    ax.set_title(full_title, fontsize=60, fontweight='bold', pad=20)

    ax.set_xlabel('Concentration (µg/m³)', fontsize=56, labelpad=15)
    ax.set_ylabel('PDF', fontsize=56, labelpad=15)
    ax.grid(True, alpha=0.3)

    if xscale == 'log':
        ax.set_xlim(left=max(1e-6, bin_edges[0]))
    if yscale == 'log':
        y_min = 1e-6
        y_max = max(n) * 10 if len(n) > 0 else 1
        ax.set_ylim(y_min, y_max)
    else:
        y_max = max(n) * 1.2 if len(n) > 0 else 1
        ax.set_ylim(0, y_max)

    gof_text = ""
    if gof_results_cf and 'anderson_darling' in gof_results_cf:
        ad = gof_results_cf['anderson_darling']
        if not np.isnan(ad['statistic']):
            gof_text += f"SSR A²={ad['statistic']:.2f} (p≈{ad['p_value']:.2f})\n"
    if gof_results_mle and 'anderson_darling' in gof_results_mle:
        ad = gof_results_mle['anderson_darling']
        if not np.isnan(ad['statistic']):
            gof_text += f"MLE A²={ad['statistic']:.2f} (p≈{ad['p_value']:.2f})"

    if gof_text.strip():
        ax.text(0.95, 0.05, gof_text, transform=ax.transAxes,
                verticalalignment='bottom', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                fontsize=48)

    ax.legend(fontsize=52, ncol=1, loc='upper right', framealpha=0.5)


def plot_pollutants_grid(results_list, location_name):
    if not results_list:
        print("No non‑ozone pollutants to plot.")
        return

    n_pollutants = len(results_list)
    if n_pollutants > 4:
        print(f"Warning: More than 4 non‑ozone pollutants found. Plotting only the first 4.")
        results_list = results_list[:4]
        n_pollutants = 4

    scales = [
        ('linear', 'linear'),
        ('linear', 'log'),
        ('log', 'log')
    ]
    titles = ['Linear-Linear', 'Log-Linear', 'Log-Log']

    fig, axes = plt.subplots(n_pollutants, 3, figsize=(60, 20 * n_pollutants))
    fig.suptitle(f"Pollutant Distributions at {location_name}",
                 fontsize=68, fontweight='bold')

    if n_pollutants == 1:
        axes = axes.reshape(1, 3)

    letters = [chr(ord('a') + i) for i in range(26)]

    for row in range(n_pollutants):
        res = results_list[row]
        for col, ((xscale, yscale), title) in enumerate(zip(scales, titles)):
            ax = axes[row, col]
            plot_on_axis(ax,
                         res['values'], res['bin_edges'],
                         res['popt_curve_fit'], res['popt_mle'],
                         res['pollutant_name'],
                         res['gof_curve_fit'], res['gof_mle'],
                         xscale=xscale, yscale=yscale, title_suffix=title)

            letter_idx = row * 3 + col
            ax.text(0.02, 0.98, f'({letters[letter_idx]})', transform=ax.transAxes,
                    fontsize=64, fontweight='bold', verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    safe_location = location_name.replace(' ', '_')
    save_path = rf"C:\Users\qp24695\Documents\{safe_location}_pollutants_grid.png"
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()


def plot_ozone_single(ozone_result, location_name):
    scales = [
        ('linear', 'linear'),
        ('linear', 'log'),
        ('log', 'log')
    ]
    titles = ['Linear-Linear', 'Log-Linear (y-log)', 'Log-Log']

    fig, axes = plt.subplots(1, 3, figsize=(60, 20))
    fig.suptitle(f"Ozone Distribution at {location_name}", fontsize=68, fontweight='bold')

    for col, ((xscale, yscale), title) in enumerate(zip(scales, titles)):
        ax = axes[col]
        plot_on_axis(ax,
                     ozone_result['values'],
                     ozone_result['bin_edges'],
                     ozone_result['popt_curve_fit'],
                     ozone_result['popt_mle'],
                     "Ozone",
                     ozone_result['gof_curve_fit'],
                     ozone_result['gof_mle'],
                     xscale=xscale, yscale=yscale, title_suffix=title)

        ax.text(0.02, 0.98, f'({chr(97+col)})', transform=ax.transAxes,
                fontsize=64, fontweight='bold', verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    safe_location = location_name.replace(' ', '_')
    save_path = rf"C:\Users\qp24695\Documents\{safe_location}_Ozone_1x3.png"
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()


def analyze_pollutant(data, pollutant_name, location_name):
    pollutant_data = data[data['Pollutant'] == pollutant_name]
    if len(pollutant_data) < 50:
        print(f"Not enough data for {pollutant_name} (n={len(pollutant_data)}). Skipping.")
        return None

    values = pollutant_data['Value'].values
    print(f"\nAnalyzing {pollutant_name} at {location_name} with {len(values)} data points")
    print(f"Data range: [{np.min(values):.2f}, {np.max(values):.2f}], Mean: {np.mean(values):.2f}, Std: {np.std(values):.2f}")

    results = {
        'pollutant_name': pollutant_name,
        'values': values,
        'bin_edges': None,
        'popt_mle': None,
        'popt_curve_fit': None,
        'gof_mle': None,
        'gof_curve_fit': None
    }

    try:
        popt_mle = fit_distribution_mle(values)
        if popt_mle is not None:
            debug_mle_fit(values, popt_mle)
            results['popt_mle'] = popt_mle
            results['gof_mle'] = perform_goodness_of_fit(values, popt_mle)
        else:
            print(f"MLE failed for {pollutant_name}.")
    except Exception as e:
        print(f"MLE error for {pollutant_name}: {str(e)}")

    try:
        popt_curve_fit, bin_centers, bin_edges, hist = fit_distribution_curve_fit(values)
        if popt_curve_fit is not None:
            results['popt_curve_fit'] = popt_curve_fit
            results['bin_edges'] = bin_edges
            results['gof_curve_fit'] = perform_goodness_of_fit(values, popt_curve_fit)
        else:
            print(f"SSR fit failed for {pollutant_name}.")
    except Exception as e:
        print(f"SSR fit error for {pollutant_name}: {str(e)}")

    if results['bin_edges'] is None:
        _, results['bin_edges'] = np.histogram(values, bins=50)

    if results['gof_curve_fit']:
        ad = results['gof_curve_fit'].get('anderson_darling', {})
        if not np.isnan(ad.get('statistic', np.nan)):
            print(f"SSR A²: {ad['statistic']:.3f}, p≈{ad['p_value']:.3f}")
    if results['gof_mle']:
        ad = results['gof_mle'].get('anderson_darling', {})
        if not np.isnan(ad.get('statistic', np.nan)):
            print(f"MLE A²: {ad['statistic']:.3f}, p≈{ad['p_value']:.3f}")

    return results


def main():
    try:
        filepath = r"C:\Users\qp24695\Downloads\Leicester University.xlsx"

        print(f"\nLoading and preprocessing data from: {filepath}")
        data = load_and_preprocess_multipollutant(filepath)

        locations = data['Location'].unique()
        if len(locations) > 1:
            print(f"Multiple locations found: {locations}. Processing each separately.")
        else:
            print(f"Location: {locations[0]}")

        pollutants = data['Pollutant'].unique()
        print(f"Pollutants to analyze: {pollutants}")

        location = locations[0]
        all_results = []
        ozone_result = None

        for pollutant in pollutants:
            res = analyze_pollutant(data[data['Location'] == location], pollutant, location)
            if res is not None:
                if pollutant.lower() == 'ozone':
                    ozone_result = res
                else:
                    all_results.append(res)

        if all_results:
            plot_pollutants_grid(all_results, location)

        if ozone_result is not None:
            plot_ozone_single(ozone_result, location)
        else:
            print("No ozone data found.")

    except Exception as e:
        print(f"Error in main: {str(e)}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import os
from tqdm import tqdm
from matplotlib import cm

plt.rcParams.update({
    'font.size': 22,
    'axes.titlesize': 22,
    'axes.labelsize': 22,
    'xtick.labelsize': 22,
    'ytick.labelsize': 22,
    'legend.fontsize': 22,
    'figure.titlesize': 22
})

def load_and_preprocess(filepath, selected_locations=None):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found at {filepath}")
    
    try:
        data = pd.read_csv(filepath, dtype=str, low_memory=False)
        data = data.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
        data.replace({'': np.nan, 'NaN': np.nan, 'NaT': np.nan}, inplace=True)
    except Exception as e:
        raise ValueError(f"Error reading CSV file: {str(e)}")
    
    if 'Date' not in data.columns or 'Time' not in data.columns:
        raise ValueError("CSV file must contain 'Date' and 'Time' columns")
    
    def fix_time(time_str):
        if time_str == '24:00:00':
            return '00:00:00'
        return time_str
    
    data['Time'] = data['Time'].apply(fix_time)
    
    try:
        datetime_str = data['Date'] + ' ' + data['Time']
        data['DateTime'] = pd.to_datetime(
            datetime_str,
            dayfirst=True,
            format='mixed',
            errors='coerce'
        )
        data = data.dropna(subset=['DateTime'])
        data.drop(['Date', 'Time'], axis=1, inplace=True)
    except Exception as e:
        raise ValueError(f"Error processing datetime: {str(e)}")
    
    location_columns = [col for col in data.columns if col not in ['DateTime']]
    
    if selected_locations:
        location_columns = [loc for loc in location_columns if loc in selected_locations]
        if not location_columns:
            raise ValueError("None of the selected locations found in data")
    
    processed_data = pd.DataFrame()
    for loc in tqdm(location_columns, desc="Processing locations"):
        if loc not in data.columns:
            print(f"Warning: Column '{loc}' not found in data")
            continue
            
        temp_df = data[['DateTime', loc]].copy()
        temp_df[loc] = temp_df[loc].astype(str).str.strip()
        
        value_extract = temp_df[loc].str.extract(r'^([\d\.-]+)')
        status_extract = temp_df[loc].str.extract(r'(V ugm-3.*|No data.*)')
        
        if value_extract.empty or status_extract.empty:
            print(f"Warning: Could not extract data for {loc}")
            continue
            
        temp_df['Value'] = value_extract[0]
        temp_df['Status'] = status_extract[0]
        
        temp_df['Value'] = pd.to_numeric(temp_df['Value'], errors='coerce')
        temp_df = temp_df.dropna(subset=['Value'])
        temp_df = temp_df[temp_df['Value'] > 0]
        
        if len(temp_df) == 0:
            print(f"Warning: No valid data for location '{loc}'")
            continue
        
        temp_df['Location'] = loc
        processed_data = pd.concat([processed_data, temp_df[['DateTime', 'Location', 'Value', 'Status']]], 
                                 ignore_index=True)
    
    if len(processed_data) < 100:
        print(f"Warning: Only {len(processed_data)} data points after preprocessing - continuing anyway")
    
    processed_data = processed_data.sort_values('DateTime')
    return processed_data

def q_gamma_pdf(x, a, q, lambda_):
    exponent = 1 / (1 - q + 1e-10) 
    term = 1 - (1 - q) * lambda_ * x
    term = np.clip(term, 1e-10, None) 
    
    return (x**(a-1)) * (term ** exponent)

def negative_log_likelihood(params, data):
    a, q, lambda_ = params
    
    penalty = 0
    if q <= 0 or q >= 2:
        penalty += 1e6 * (abs(q - 1) + 1)  
    
    pdf_values = q_gamma_pdf(data, a, q, lambda_)

    x_range = np.linspace(min(data), max(data), 1000)
    pdf_norm = q_gamma_pdf(x_range, a, q, lambda_)
    norm = np.trapz(pdf_norm, x_range)

    epsilon = 1e-10
    normalized_pdf = pdf_values / (norm + epsilon)
    log_pdf = np.log(normalized_pdf + epsilon)
    
    return -np.sum(log_pdf) + penalty

def fit_distribution(values, num_bins=200):

    initial_params = [2, 1.0, 0.1] 
    bounds = [(0.001, 15), (0, 2), (0.001, 150)] 
    
    try:
        result = minimize(
            negative_log_likelihood,
            initial_params,
            args=(values,),
            bounds=bounds,
            method='L-BFGS-B',
            options={'maxiter': 2000, 'disp': True}
        )
        
        if not result.success:
            print(f"Optimization failed: {result.message}")
            return None, None, None, None

        hist, bin_edges = np.histogram(values, bins=num_bins, density=True)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        
        return result.x, bin_centers, bin_edges, hist
    
    except Exception as e:
        print(f"Error in MLE fitting: {str(e)}")
        return None, None, None, None

def analyze_location(data, location_name):
    location_data = data[data['Location'] == location_name]
    if len(location_data) < 20:  
        print(f"Not enough data for {location_name} (only {len(location_data)} points)")
        return None
    
    values = location_data['Value'].values
    popt_gamma, _, _, _ = fit_distribution(values)
    
    return popt_gamma

def get_seasonal_data(data, season):
    """Extract data for specific season"""
    if season == 'winter':
        seasonal_data = data[data['DateTime'].dt.month.isin([12, 1, 2])]
    elif season == 'summer':
        seasonal_data = data[data['DateTime'].dt.month.isin([6, 7, 8])]
    else:
        seasonal_data = data.copy()
    
    return seasonal_data

def get_day_night_data(seasonal_data, season, period):
    if season == 'winter':
        if period == 'night':
            mask = (seasonal_data['DateTime'].dt.hour >= 19) | (seasonal_data['DateTime'].dt.hour < 6)
        elif period == 'day':
            mask = (seasonal_data['DateTime'].dt.hour >= 8) & (seasonal_data['DateTime'].dt.hour < 16)
        else:
            return seasonal_data
    
    elif season == 'summer':
        if period == 'night':
            mask = (seasonal_data['DateTime'].dt.hour >= 23) | (seasonal_data['DateTime'].dt.hour < 4)
        elif period == 'day':
            mask = (seasonal_data['DateTime'].dt.hour >= 6) & (seasonal_data['DateTime'].dt.hour < 21)
        else:
            return seasonal_data
    else:
        return seasonal_data
    
    return seasonal_data[mask]

def analyze_seasonal_day_night_data(data, data_type, season, period):
    parameter_data = {}

    seasonal_data = get_seasonal_data(data, season)

    period_data = get_day_night_data(seasonal_data, season, period)
    
    if len(period_data) == 0:
        print(f"No data for {data_type} - {season} {period}")
        return parameter_data
    
    locations = period_data['Location'].unique()
    
    print(f"Analyzing {data_type} - {season.capitalize()} {period} ({len(period_data)} data points)")
    
    for location in locations:
        location_data = period_data[period_data['Location'] == location]
        if len(location_data) < 15:  
            continue
            
        params = analyze_location(period_data, location)
        if params is not None:
            parameter_data[location] = params
            print(f"{data_type} - {location} ({season} {period}): a={params[0]:.4f}, q={params[1]:.4f}, λ={params[2]:.4f}")
    
    return parameter_data

def plot_all_pollutants_vertical(all_pollutant_results, scenario_key="all", scenario_name="All Data"):

    pollutants = ['NO₂', 'NO', 'PM₁₀', 'PM₂.₅']  
    pollutants_keys = ['NO2', 'NO', 'PM10', 'PM2.5']  
    site_types = ['Urban Traffic', 'Urban Industrial', 'Urban Background', 
                  'Suburban Industrial', 'Suburban Background', 'Rural Background']
    
    colors = cm.viridis(np.linspace(0, 1, len(site_types)))
    site_type_colors = dict(zip(site_types, colors))

    fig, axes = plt.subplots(len(pollutants), 3, figsize=(18, 7*len(pollutants)))
    
    for i, (pollutant_display, pollutant_key) in enumerate(zip(pollutants, pollutants_keys)):
        if pollutant_key not in all_pollutant_results or scenario_key not in all_pollutant_results[pollutant_key]:
            print(f"No data for {pollutant_key} in scenario {scenario_name}")
            continue
            
        param_data = all_pollutant_results[pollutant_key][scenario_key]

        ax = axes[i, 0]
        for data_type, parameter_data in param_data.items():
            if not parameter_data:
                continue
            locations = list(parameter_data.keys())
            a_values = [parameter_data[loc][0] for loc in locations if parameter_data[loc] is not None]
            q_values = [parameter_data[loc][1] for loc in locations if parameter_data[loc] is not None]
            ax.scatter(a_values, q_values, alpha=0.8, c=[site_type_colors[data_type]], 
                      label=data_type, s=60, edgecolors='k', linewidth=0.5)
        
        ax.set_xlabel('α')
        ax.set_ylabel('q')
        ax.set_title(f'{pollutant_display}', fontweight='bold')
        ax.axhline(y=1, color='gray', linestyle='--', alpha=0.7, linewidth=1.5)
        ax.axvline(x=1, color='gray', linestyle='--', alpha=0.7, linewidth=1.5)
        ax.grid(True, alpha=0.3, linestyle='--')

        ax.text(0.02, 0.98, f'({chr(97 + i*3)})', transform=ax.transAxes, 
                fontsize=22, fontweight='bold', va='top', ha='left')

        ax = axes[i, 1]
        for data_type, parameter_data in param_data.items():
            if not parameter_data:
                continue
            locations = list(parameter_data.keys())
            lambda_values = [parameter_data[loc][2] for loc in locations if parameter_data[loc] is not None]
            q_values = [parameter_data[loc][1] for loc in locations if parameter_data[loc] is not None]
            ax.scatter(lambda_values, q_values, alpha=0.8, c=[site_type_colors[data_type]], 
                      label=data_type, s=60, edgecolors='k', linewidth=0.5)
        
        ax.set_xscale('log')
        ax.set_xlabel('λ')
        ax.set_ylabel('q')
        ax.axhline(y=1, color='gray', linestyle='--', alpha=0.7, linewidth=1.5)
        ax.grid(True, alpha=0.3, linestyle='--')

        ax.text(0.02, 0.98, f'({chr(97 + i*3 + 1)})', transform=ax.transAxes, 
                fontsize=22, fontweight='bold', va='top', ha='left')

        ax = axes[i, 2]
        for data_type, parameter_data in param_data.items():
            if not parameter_data:
                continue
            locations = list(parameter_data.keys())
            lambda_values = [parameter_data[loc][2] for loc in locations if parameter_data[loc] is not None]
            a_values = [parameter_data[loc][0] for loc in locations if parameter_data[loc] is not None]
            ax.scatter(lambda_values, a_values, alpha=0.8, c=[site_type_colors[data_type]], 
                      label=data_type, s=60, edgecolors='k', linewidth=0.5)
        
        ax.set_xscale('log')
        ax.set_xlabel('λ')
        ax.set_ylabel('α')
        ax.grid(True, alpha=0.3, linestyle='--')

        ax.text(0.02, 0.98, f'({chr(97 + i*3 + 2)})', transform=ax.transAxes, 
                fontsize=22, fontweight='bold', va='top', ha='left')

    legend_elements = []
    for site_type, color in site_type_colors.items():
        legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', 
                                         markerfacecolor=color, markersize=12,
                                         label=site_type, markeredgecolor='k'))

    fig.legend(handles=legend_elements, loc='lower center', 
               bbox_to_anchor=(0.5, 0.02), ncol=3, frameon=True,
               fancybox=True, shadow=True, borderpad=1)

    plt.tight_layout(rect=[0, 0.08, 1, 0.98])  
    fig.suptitle(f'Parameter Relationships for Air Pollutants ({scenario_name})', 
                 fontsize=24, fontweight='bold', y=0.99)

    filename_suffix = scenario_name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(fr"C:\Users\qp24695\Documents\all_pollutants_vertical_{filename_suffix}.png", 
                dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.show()

    plot_q_vs_alpha_comparison(all_pollutant_results, scenario_key, scenario_name)

def plot_q_vs_alpha_comparison(all_pollutant_results, scenario_key="all", scenario_name="All Data"):

    pollutants_display = ['NO₂', 'NO', 'PM₁₀', 'PM₂.₅']
    pollutants_keys = ['NO2', 'NO', 'PM10', 'PM2.5']

    site_types = ['Urban Traffic', 'Urban Industrial', 'Urban Background', 
                  'Suburban Industrial', 'Suburban Background', 'Rural Background']
    colors = cm.viridis(np.linspace(0, 1, len(site_types)))
    site_type_colors = dict(zip(site_types, colors))

    fig, ax = plt.subplots(figsize=(16, 12))

    for poll_idx, (pollutant_display, pollutant_key) in enumerate(zip(pollutants_display, pollutants_keys)):
        if pollutant_key not in all_pollutant_results or scenario_key not in all_pollutant_results[pollutant_key]:
            continue
            
        param_data = all_pollutant_results[pollutant_key][scenario_key]
        
        for data_type, parameter_data in param_data.items():
            if not parameter_data:
                continue
            
            locations = list(parameter_data.keys())
            a_values = [parameter_data[loc][0] for loc in locations if parameter_data[loc] is not None]
            q_values = [parameter_data[loc][1] for loc in locations if parameter_data[loc] is not None]
            
            ax.scatter(a_values, q_values, alpha=0.7, c=[site_type_colors[data_type]], 
                      s=80, edgecolors='k', linewidth=0.8,
                      label=f'{pollutant_display} - {data_type}' if poll_idx == 0 else '')
    
    ax.set_xlabel('α')
    ax.set_ylabel('q')
    ax.set_title(f'q vs α for All Pollutants ({scenario_name})', fontweight='bold')
    ax.axhline(y=1, color='gray', linestyle='--', alpha=0.7, linewidth=2)
    ax.axvline(x=1, color='gray', linestyle='--', alpha=0.7, linewidth=2)
    ax.grid(True, alpha=0.3, linestyle='--')

    legend_elements = []
    for site_type, color in site_type_colors.items():
        legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', 
                                         markerfacecolor=color, markersize=12,
                                         label=site_type, markeredgecolor='k'))
    
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.05, 1),
              frameon=True, fancybox=True, shadow=True)
    
    plt.tight_layout()
    plt.savefig(fr"C:\Users\qp24695\Documents\q_vs_alpha_comparison_{scenario_name.lower().replace(' ', '_')}.png", 
                dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.show()

def analyze_pollutant(pollutant_name, filepaths, selected_locations=None):
    print(f"\n{'='*80}")
    print(f"ANALYZING {pollutant_name.upper()} - ALL DATA ONLY")
    print(f"{'='*80}")

    all_scenarios_parameter_data = {'all': {}}

    for data_type, filepath in filepaths.items():
        print(f"\nProcessing {data_type} data for {pollutant_name}...")
        
        try:
            if not os.path.exists(filepath):
                print(f"File not found: {filepath}")
                continue

            data = load_and_preprocess(filepath, selected_locations)

            print(f"--- Analyzing ALL DATA for {data_type} ---")
            all_scenarios_parameter_data['all'][data_type] = {}
            locations = data['Location'].unique()
            
            for location in locations:
                params = analyze_location(data, location)
                if params is not None:
                    all_scenarios_parameter_data['all'][data_type][location] = params
                    print(f"  {location}: a={params[0]:.3f}, q={params[1]:.3f}, λ={params[2]:.3f}")
        
        except Exception as e:
            print(f"Error processing {data_type} data for {pollutant_name}: {str(e)}")
            continue
    
    return all_scenarios_parameter_data

def main():
    try:
        pollutant_configs = {
            'NO2': {
                'Urban Traffic': r"C:\Users\qp24695\Downloads\NO2UrbanTraUK.csv",
                'Urban Industrial': r"C:\Users\qp24695\Downloads\NO2UrbanInUK.csv",
                'Urban Background': r"C:\Users\qp24695\Downloads\NO2UrbanBackUK.csv",
                'Suburban Industrial': r"C:\Users\qp24695\Downloads\NO2SubInUK.csv",
                'Suburban Background': r"C:\Users\qp24695\Downloads\NO2SubUK.csv",
                'Rural Background': r"C:\Users\qp24695\Downloads\NO2RuralUK.csv",
            },
            'NO': {
                'Urban Traffic': r"C:\Users\qp24695\Downloads\NOUrbanTraUK.csv",
                'Urban Industrial': r"C:\Users\qp24695\Downloads\NOUrbanInUK.csv",
                'Urban Background': r"C:\Users\qp24695\Downloads\NOUrbanBackUK.csv",
                'Suburban Industrial': r"C:\Users\qp24695\Downloads\NOSubInUK.csv",
                'Suburban Background': r"C:\Users\qp24695\Downloads\NOSubUK.csv",
                'Rural Background': r"C:\Users\qp24695\Downloads\NORuralUK.csv",
            },
            'PM10': {
                'Urban Traffic': r"C:\Users\qp24695\Downloads\pm10UrbanTraUK.csv",
                'Urban Industrial': r"C:\Users\qp24695\Downloads\pm105UrbanInUK.csv",
                'Urban Background': r"C:\Users\qp24695\Downloads\pm10UrbanBackUK.csv",
                'Suburban Industrial': r"C:\Users\qp24695\Downloads\pm10SubInUK.csv",
                'Suburban Background': r"C:\Users\qp24695\Downloads\pm10SubUK.csv",
                'Rural Background': r"C:\Users\qp24695\Downloads\pm10RuralUK.csv",
            },
            'PM2.5': {
                'Urban Traffic': r"C:\Users\qp24695\Downloads\pm2.5UrbanTraUK.csv",
                'Urban Industrial': r"C:\Users\qp24695\Downloads\pm2.5UrbanInUK.csv",
                'Urban Background': r"C:\Users\qp24695\Downloads\pm2.5UrbanBackUK.csv",
                'Suburban Industrial': r"C:\Users\qp24695\Downloads\pm2.5SubInUK.csv",
                'Suburban Background': r"C:\Users\qp24695\Downloads\pm2.5SubUK.csv",
                'Rural Background': r"C:\Users\qp24695\Downloads\pm2.5RuralUK.csv",
            }
        }
        
        all_pollutant_results = {}

        for pollutant_name, filepaths in pollutant_configs.items():
            results = analyze_pollutant(pollutant_name, filepaths)
            all_pollutant_results[pollutant_name] = results

        print(f"\n{'='*80}")
        print("GENERATING COMPREHENSIVE VERTICAL PLOT")
        print(f"{'='*80}")
        
        plot_all_pollutants_vertical(all_pollutant_results, scenario_key="all", scenario_name="All Data")

        print("\n" + "="*100)
        print("SUMMARY STATISTICS FOR ALL POLLUTANTS (All Data)")
        print("="*100)
        
        for pollutant_key, results in all_pollutant_results.items():
            if 'all' in results:
                print(f"\n{pollutant_key}:")
                
                for data_type, param_data in results['all'].items():
                    if param_data:
                        num_locations = len(param_data)

                        a_values = [param_data[loc][0] for loc in param_data if param_data[loc] is not None]
                        q_values = [param_data[loc][1] for loc in param_data if param_data[loc] is not None]
                        lambda_values = [param_data[loc][2] for loc in param_data if param_data[loc] is not None]
                        
                        if a_values:
                            print(f"  {data_type:<25} {num_locations:>2} locations")
                            print(f"    α: {np.mean(a_values):.3f} ± {np.std(a_values):.3f}")
                            print(f"    q: {np.mean(q_values):.3f} ± {np.std(q_values):.3f}")
                            print(f"    λ: {np.mean(lambda_values):.3f} ± {np.std(lambda_values):.3f}")
    
    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf
from scipy.optimize import curve_fit
from scipy.stats import anderson, kstest
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 22,
    'axes.titlesize': 22,
    'axes.labelsize': 22,
    'xtick.labelsize': 22,
    'ytick.labelsize': 22,
    'legend.fontsize': 22,
    'figure.titlesize': 22
})

def q_exp(x, q, λ):
    return np.where(1 + (q - 1) * λ * x > 0,
                   (1 + (q - 1) * λ * x) ** (1 / (1 - q)),
                   np.nan)

df = pd.read_csv(r"C:\Users\qp24695\Documents\Air\data\London Marylebone Road - Westminster (Urban Traffic).csv")
df['Time'] = df['Time'].apply(lambda x: '00:00:00' if x == '24:00:00' else x)
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
df = df.dropna(subset=['Datetime']).set_index('Datetime')

def analyze_period(data, title):
    numeric_data = data[pollutants].apply(pd.to_numeric, errors='coerce')
    results = []
    plot_data = {}
    
    for pollutant in pollutants:
        series = numeric_data[pollutant].dropna()
        if len(series) < lags:
            continue
            
        autocorr = acf(series, nlags=lags, fft=True)
        normalized_autocorr = autocorr / autocorr[0]
        x_data = np.arange(lags + 1)
        y_data = normalized_autocorr
        
        try:
            popt, pcov = curve_fit(q_exp, x_data, y_data, p0=[1.1, 0.1], maxfev=5000)
            q, λ = popt
            q_exp_fit = q_exp(x_data, q, λ)

            residuals = y_data - q_exp_fit
            norm_residuals = residuals / np.std(residuals)
            ad_result = anderson(norm_residuals)
            ad_stat = ad_result.statistic
            ad_critical = ad_result.critical_values[2]
            ks_stat, ks_p = kstest(y_data, lambda x: q_exp(x, q, λ))
            
            results.append({
                'Pollutant': pollutant,
                'q': q,
                'lambda': λ,
                'AD_stat': ad_stat,
                'AD_critical_5%': ad_critical,
                'KS_stat': ks_stat,
                'KS_pvalue': ks_p,
                'Fit_Valid': (ad_stat < ad_critical) and (ks_p > 0.05)
            })
            
            plot_data[pollutant] = {
                'x': x_data,
                'y': y_data,
                'fit': q_exp_fit,
                'q': q,
                'λ': λ
            }
            
        except Exception as e:
            print(f"Error fitting {pollutant}: {str(e)}")
    
    return pd.DataFrame(results), plot_data, title

pollutants = ['Nitric oxide', 'Nitrogen dioxide', 'PM10 particulate matter',
              'PM2.5 particulate matter']
lags = 5

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

night_mask = (df.index.hour >= 18) | (df.index.hour < 5)
night_results, night_plot_data, night_title = analyze_period(df[night_mask], "Nighttime (18:00-04:59)")

day_mask = (df.index.hour >= 6) & (df.index.hour < 17)
day_results, day_plot_data, day_title = analyze_period(df[day_mask], "Daytime (06:00-17:59)")

colors = plt.cm.Set1(np.linspace(0, 1, len(pollutants)))

ax1 = axes[0]
for idx, pollutant in enumerate(pollutants):
    if pollutant in night_plot_data:
        data = night_plot_data[pollutant]
        ax1.scatter(data['x'], data['y'], color=colors[idx], alpha=0.6, s=50, edgecolor='none')
        ax1.plot(data['x'], data['fit'], '-', color=colors[idx], linewidth=2.5, alpha=0.8)

ax1.text(0.02, 0.98, '(a)', transform=ax1.transAxes, fontsize=24, fontweight='bold', verticalalignment='top')
ax1.set_xlabel('Lag (hours)', fontsize=22)
ax1.set_ylabel('Normalized Autocorrelation', fontsize=22)
ax1.set_title(f'{night_title}\nLondon Marylebone Road', fontsize=22, fontweight='bold')
ax1.axhline(0, color='gray', linestyle='-', alpha=0.7)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')
ax1.tick_params(axis='both', which='major', labelsize=20)

night_param_text = ""
for pollutant in pollutants:
    if pollutant in night_plot_data:
        q = night_plot_data[pollutant]['q']
        lam = night_plot_data[pollutant]['λ']
        short_name = pollutant.replace(' particulate matter', '').replace('Nitric oxide', 'NO').replace('Nitrogen dioxide', 'NO2')
        night_param_text += f"{short_name}: q={q:.3f}, λ={lam:.3f}\n"
ax1.text(0.95, 0.95, night_param_text, transform=ax1.transAxes, fontsize=16,
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax2 = axes[1]
for idx, pollutant in enumerate(pollutants):
    if pollutant in day_plot_data:
        data = day_plot_data[pollutant]
        ax2.scatter(data['x'], data['y'], color=colors[idx], alpha=0.6, s=50, edgecolor='none')
        ax2.plot(data['x'], data['fit'], '-', color=colors[idx], linewidth=2.5, alpha=0.8)

ax2.text(0.02, 0.98, '(b)', transform=ax2.transAxes, fontsize=24, fontweight='bold', verticalalignment='top')
ax2.set_xlabel('Lag (hours)', fontsize=22)
ax2.set_ylabel('Normalized Autocorrelation', fontsize=22)
ax2.set_title(f'{day_title}\nLondon Marylebone Road', fontsize=22, fontweight='bold')
ax2.axhline(0, color='gray', linestyle='-', alpha=0.7)
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')
ax2.tick_params(axis='both', which='major', labelsize=20)

day_param_text = ""
for pollutant in pollutants:
    if pollutant in day_plot_data:
        q = day_plot_data[pollutant]['q']
        lam = day_plot_data[pollutant]['λ']
        short_name = pollutant.replace(' particulate matter', '').replace('Nitric oxide', 'NO').replace('Nitrogen dioxide', 'NO2')
        day_param_text += f"{short_name}: q={q:.3f}, λ={lam:.3f}\n"
ax2.text(0.95, 0.95, day_param_text, transform=ax2.transAxes, fontsize=16,
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

lines = []
labels = []
for idx, pollutant in enumerate(pollutants):
    if pollutant in night_plot_data or pollutant in day_plot_data:
        line = plt.Line2D([0], [0], color=colors[idx], linestyle='-', linewidth=2.5)
        lines.append(line)
        labels.append(pollutant)

fig.legend(lines, labels, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.05), fontsize=20,
           frameon=True, fancybox=True, shadow=True, borderpad=1)

plt.suptitle('Q-Exponential Fit Comparison', fontsize=24, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.1, 1, 0.95])
plt.savefig(r"C:\Users\qp24695\Documents\qexp_fit_comparison.png",
            bbox_inches='tight', dpi=300)
plt.show()

print("\n" + "="*80)
print("NIGHTTIME RESULTS (18:00-04:59)")
print("="*80)
print(night_results.to_string(index=False))

print("\n" + "="*80)
print("DAYTIME RESULTS (06:00-17:59)")
print("="*80)
print(day_results.to_string(index=False))

summary_df = pd.DataFrame({
    'Pollutant': pollutants,
    'Night_q': night_results.set_index('Pollutant').reindex(pollutants)['q'].values,
    'Day_q': day_results.set_index('Pollutant').reindex(pollutants)['q'].values,
    'Night_λ': night_results.set_index('Pollutant').reindex(pollutants)['lambda'].values,
    'Day_λ': day_results.set_index('Pollutant').reindex(pollutants)['lambda'].values,
    'Night_Fit_Valid': night_results.set_index('Pollutant').reindex(pollutants)['Fit_Valid'].values,
    'Day_Fit_Valid': day_results.set_index('Pollutant').reindex(pollutants)['Fit_Valid'].values
})

print("\n" + "="*80)
print("SUMMARY COMPARISON")
print("="*80)
print(summary_df.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf
from scipy.optimize import curve_fit
from scipy.stats import chi2
import os
from matplotlib import cm
from matplotlib.lines import Line2D

def q_exp(x, q, λ):
    return np.where(1 + (q - 1) * λ * x > 0, 
                   (2 - q) * (1 + (q - 1) * λ * x) ** (1 / (1 - q)), 
                   np.nan)


parameter_data_dict = {
    'Urban Traffic': {},
    'Urban Industrial': {},
    'Urban Background': {},
    'Suburban Industrial': {},
    'Suburban Background': {},
    'Rural Background': {},
}

filepaths_dict = {
    'NO': {
        'Urban Traffic': r"C:\Users\qp24695\Downloads\NOUrbanTraUK.csv",
        'Urban Industrial': r"C:\Users\qp24695\Downloads\NOUrbanInUK.csv",
        'Urban Background': r"C:\Users\qp24695\Downloads\NOUrbanBackUK.csv",
        'Suburban Industrial': r"C:\Users\qp24695\Downloads\NOSubInUK.csv",
        'Suburban Background': r"C:\Users\qp24695\Downloads\NOSubUK.csv",
        'Rural Background': r"C:\Users\qp24695\Downloads\NORuralUK.csv", 
    },
    'NO2': {
        'Urban Traffic': r"C:\Users\qp24695\Downloads\NO2UrbanTraUK.csv",
        'Urban Industrial': r"C:\Users\qp24695\Downloads\NO2UrbanInUK.csv",
        'Urban Background': r"C:\Users\qp24695\Downloads\NO2UrbanBackUK.csv",
        'Suburban Industrial': r"C:\Users\qp24695\Downloads\NO2SubInUK.csv",
        'Suburban Background': r"C:\Users\qp24695\Downloads\NO2SubUK.csv",
        'Rural Background': r"C:\Users\qp24695\Downloads\NO2RuralUK.csv", 
    },
    'PM2.5': {
        'Urban Traffic': r"C:\Users\qp24695\Downloads\pm2.5UrbanTraUK.csv",
        'Urban Industrial': r"C:\Users\qp24695\Downloads\pm2.5UrbanInUK.csv",
        'Urban Background': r"C:\Users\qp24695\Downloads\pm2.5UrbanBackUK.csv",
        'Suburban Industrial': r"C:\Users\qp24695\Downloads\pm2.5SubInUK.csv",
        'Suburban Background': r"C:\Users\qp24695\Downloads\pm2.5SubUK.csv",
        'Rural Background': r"C:\Users\qp24695\Downloads\pm2.5RuralUK.csv", 
    },
    'PM10': {
        'Urban Traffic': r"C:\Users\qp24695\Downloads\pm10UrbanTraUK.csv",
        'Urban Industrial': r"C:\Users\qp24695\Downloads\pm10UrbanInUK.csv",
        'Urban Background': r"C:\Users\qp24695\Downloads\pm10UrbanBackUK.csv",
        'Suburban Industrial': r"C:\Users\qp24695\Downloads\pm10SubInUK.csv",
        'Suburban Background': r"C:\Users\qp24695\Downloads\pm10SubUK.csv",
        'Rural Background': r"C:\Users\qp24695\Downloads\pm10RuralUK.csv", 
    }
}

output_dir = r"C:\Users\qp24695\Documents\Air\analysis_results"
os.makedirs(output_dir, exist_ok=True)

all_pollutant_results = {}

for pollutant_name, filepaths in filepaths_dict.items():
    print(f"Processing {pollutant_name}...")
    all_results = []

    for site_type, filepath in filepaths.items():
        try:
            df = pd.read_csv(filepath)

            df['Time'] = df['Time'].apply(lambda x: '00:00:00' if x == '24:00:00' else x)
            df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
            df = df.dropna(subset=['Datetime']).set_index('Datetime')
            site_columns = df.columns[2::2]
            site_names = [col.strip() for col in site_columns]

            for i, site in enumerate(site_names):

                value_col = df.columns[2 + i*2]
                
                site_df = pd.DataFrame({
                    'Datetime': df.index,
                    'Value': pd.to_numeric(df[value_col], errors='coerce')
                }).dropna().set_index('Datetime')
                
                if len(site_df) < 10:  
                    continue

                nighttime_mask = (site_df.index.hour >= 18) | (site_df.index.hour < 5)
                nighttime_series = site_df[nighttime_mask]['Value'].dropna()

                daytime_mask = (site_df.index.hour >= 6) & (site_df.index.hour < 17)
                daytime_series = site_df[daytime_mask]['Value'].dropna()

                def process_series(series, period_name, min_lags):
                    if len(series) < min_lags:
                        return None
                    
                    autocorr = acf(series, nlags=min_lags-1, fft=True)
                    normalized_autocorr = autocorr / autocorr[0]
                    x_data = np.arange(len(autocorr))
                    
                    try:
                        popt, pcov = curve_fit(q_exp, x_data, normalized_autocorr, p0=[1.1, 0.1], maxfev=5000)

                        predicted = q_exp(x_data, *popt)
                        residuals = normalized_autocorr - predicted
                        chi_square = np.sum((residuals**2) / predicted)
                        degrees_of_freedom = len(x_data) - len(popt) - 1
                        p_value = 1 - chi2.cdf(chi_square, degrees_of_freedom)
                        
                        return {
                            'Site': site,
                            'Site Type': site_type,
                            'Period': period_name,
                            'q': popt[0],
                            'λ': popt[1],
                            'Data Points': len(series),
                            'Chi-Square': chi_square,
                            'Degrees of Freedom': degrees_of_freedom,
                            'p-value': p_value,
                            'Fit Quality': 'Good' if p_value > 0.05 else 'Poor'
                        }
                    except Exception as e:
                        print(f"Error fitting {site} {period_name}: {str(e)}")
                        return None

                night_result = process_series(nighttime_series, 'Night', 6)
                day_result = process_series(daytime_series, 'Day', 6)
                
                if night_result:
                    all_results.append(night_result)
                if day_result:
                    all_results.append(day_result)
                    
        except Exception as e:
            print(f"Error processing {site_type} for {pollutant_name}: {str(e)}")
            continue

    results_df = pd.DataFrame(all_results)
    all_pollutant_results[pollutant_name] = results_df

    results_df.to_csv(os.path.join(output_dir, f'q_lambda_results_{pollutant_name}.csv'), index=False)
    
    print(f"  {pollutant_name} complete: {len(results_df)} data points")

site_types = list(parameter_data_dict.keys())
colors = cm.viridis(np.linspace(0, 1, len(site_types)))
site_type_colors = dict(zip(site_types, colors))

fig, axes = plt.subplots(2, 2, figsize=(22, 18))
axes = axes.flatten()

plt.rcParams.update({
    'font.size': 20,           
    'axes.titlesize': 24,      
    'axes.labelsize': 22,      
    'legend.fontsize': 16,     
    'xtick.labelsize': 20,     
    'ytick.labelsize': 20,     
    'figure.titlesize': 28,    
})


pollutants = ['NO', 'NO2', 'PM2.5', 'PM10']
markers = {'Day': 'o', 'Night': 's'}

for idx, pollutant in enumerate(pollutants):
    ax = axes[idx]
    results_df = all_pollutant_results.get(pollutant)
    
    if results_df is None or len(results_df) == 0:
        ax.text(0.5, 0.5, f'No data for {pollutant}', 
                ha='center', va='center', transform=ax.transAxes, fontsize=22)
        ax.set_xlabel('q value', fontsize=22)
        ax.set_ylabel('λ value', fontsize=22)
        continue
    
    for site_type in site_types:
        type_data = results_df[results_df['Site Type'] == site_type]
        for period in ['Day', 'Night']:
            period_data = type_data[type_data['Period'] == period]
            if not period_data.empty:
                ax.scatter(
                    period_data['q'],
                    period_data['λ'],
                    c=[site_type_colors[site_type]] * len(period_data),
                    marker=markers[period],
                    s=150,
                    alpha=0.7,
                    edgecolors='w',
                    linewidth=1.0,
                    label=f'{site_type} ({period})' if idx == 0 else ''
                )
    
    poor_fits = results_df[results_df['Fit Quality'] == 'Poor']
    for site_type in site_types:
        type_data = poor_fits[poor_fits['Site Type'] == site_type]
        for period in ['Day', 'Night']:
            period_data = type_data[type_data['Period'] == period]
            if not period_data.empty:
                ax.scatter(
                    period_data['q'],
                    period_data['λ'],
                    c='red',
                    marker='x',
                    s=100,  
                    linewidth=3.0,  
                    alpha=0.9,
                    label='Poor fit' if idx == 0 else ''
                )
    
    ax.set_xlabel('q value', fontsize=22, fontweight='bold')
    ax.set_ylabel('λ value', fontsize=22, fontweight='bold')
    ax.set_title(f'{pollutant} Autocorrelation fit', fontsize=24, fontweight='bold')
    ax.grid(True, alpha=0.3)

    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='minor', labelsize=18)
    

    label_pos = (-0.12, 1.05) if idx < 2 else (-0.12, 1.02)
    ax.text(*label_pos, f'({chr(97+idx)})', transform=ax.transAxes, 
            fontsize=32, fontweight='extra bold', va='top')

legend_ax = axes[0]
site_legend_elements = [Line2D([0], [0], marker='o', color='w', label=site_type,
                         markerfacecolor=site_type_colors[site_type], markersize=16)
                  for site_type in site_types]

period_legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Daytime',
          markerfacecolor='gray', markersize=16),
    Line2D([0], [0], marker='s', color='w', label='Nighttime',
          markerfacecolor='gray', markersize=16),
    Line2D([0], [0], marker='x', color='red', label='Poor fit',
          markersize=16, linewidth=3.0)
]

first_site_legend = legend_ax.legend(handles=site_legend_elements, title='Site Types', 
                                    loc='upper right', fontsize=16, title_fontsize=18)
legend_ax.add_artist(first_site_legend)
legend_ax.legend(handles=period_legend_elements, title='Time Periods & Fit Quality', 
                loc='lower right', fontsize=16, title_fontsize=18)

plt.tight_layout(pad=3.0, h_pad=4.0, w_pad=4.0)
plt.savefig(os.path.join(output_dir, 'all_pollutants_q_vs_lambda_combined.png'), 
           dpi=300, bbox_inches='tight')

for pollutant in pollutants:
    results_df = all_pollutant_results.get(pollutant)
    if results_df is None or len(results_df) == 0:
        continue
    
    plt.figure(figsize=(14, 12))
    ax = plt.gca()

    plt.rcParams.update({
        'font.size': 18,
        'axes.titlesize': 22,
        'axes.labelsize': 20,
        'legend.fontsize': 16,
        'xtick.labelsize': 18,
        'ytick.labelsize': 18,
    })
    

    for site_type in site_types:
        type_data = results_df[results_df['Site Type'] == site_type]
        for period in ['Day', 'Night']:
            period_data = type_data[type_data['Period'] == period]
            if not period_data.empty:
                ax.scatter(
                    period_data['q'],
                    period_data['λ'],
                    c=[site_type_colors[site_type]] * len(period_data),
                    marker=markers[period],
                    s=150,
                    alpha=0.7,
                    edgecolors='w',
                    linewidth=1.0,
                    label=f'{site_type} ({period})'
                )
    
    poor_fits = results_df[results_df['Fit Quality'] == 'Poor']
    for site_type in site_types:
        type_data = poor_fits[poor_fits['Site Type'] == site_type]
        for period in ['Day', 'Night']:
            period_data = type_data[type_data['Period'] == period]
            if not period_data.empty:
                ax.scatter(
                    period_data['q'],
                    period_data['λ'],
                    c='red',
                    marker='x',
                    s=100,
                    linewidth=3.0,
                    alpha=0.9,
                    label='_nolegend_'
                )

    ax.set_xlabel('q value', fontsize=20, fontweight='bold')
    ax.set_ylabel('λ value', fontsize=20, fontweight='bold')
    ax.set_title(f'{pollutant} Autocorrelation fit Q vs λ', fontsize=22, fontweight='bold')
    ax.grid(True, alpha=0.3)

    ax.tick_params(axis='both', which='major', labelsize=18)
    ax.tick_params(axis='both', which='minor', labelsize=16)

    legend_elements = []
    for site_type in site_types[:3]:  
        legend_elements.append(Line2D([0], [0], marker='o', color='w', label=site_type,
                                    markerfacecolor=site_type_colors[site_type], markersize=16))
    
    legend_elements.extend([
        Line2D([0], [0], marker='o', color='w', label='Daytime',
              markerfacecolor='gray', markersize=16),
        Line2D([0], [0], marker='s', color='w', label='Nighttime',
              markerfacecolor='gray', markersize=16),
        Line2D([0], [0], marker='x', color='red', label='Poor fit',
              markersize=16, linewidth=3.0)
    ])
    
    ax.legend(handles=legend_elements, title='Site Types & Time Period', 
             fontsize=16, title_fontsize=18, loc='best')
    
    plt.tight_layout(pad=3.0)
    plt.savefig(os.path.join(output_dir, f'{pollutant}_q_vs_lambda_individual.png'), 
               dpi=300, bbox_inches='tight')
    plt.close()

print(f"\nAnalysis complete. All results saved to {output_dir}")
print(f"Main combined plot saved as: all_pollutants_q_vs_lambda_combined.png")
print(f"Individual plots and CSV files also saved for each pollutant.")